# USAGE EXAMPLE FOR FARMWISE API - Python

## Specify you login, password and payload

In [ ]:
import requests

# Define the login URL and the data endpoint URL
login_url = "https://a52cbcc62931.ngrok-free.app/token"
data_url = "https://a52cbcc62931.ngrok-free.app/read-data-direct"

# User credentials
username = "login"
password = "pass"

# Define the request payload
data = {
    "bounding_box": [51.09, 50.00, 14.56, 14.14],
    "level": 10,
    "time_from": '2018-01-01',
    "time_to": '2018-02-01',
    "factors": ["precipitation", "temperature"],
    "separate_api": True,
    "interpolation": False
}


## Code helpers

In [ ]:
def login(username, password):
    # Prepare the payload for the login request
    payload = {
        "username": username,
        "password": password
    }

    # Send a POST request to the login endpoint
    response = requests.post(login_url, data=payload)

    if response.status_code == 200:
        # Extract the access token from the response
        token = response.json().get("access_token")
        return token
    else:
        print(f"Failed to login: {response.status_code}")
        print(response.json())
        return None

In [ ]:
import re

def fetch_data(token):
    # Define the headers with the JWT token
    headers = {
        "Authorization": f"Bearer {token}"
    }
    # Send a POST request to the data endpoint
    response = requests.post(data_url, json=data, headers=headers)

    # Handle the response
    if response.status_code == 200:
        # Collect the streamed response
        response_text = response.text
        print("Response received:")
        print(response_text)

        # Extract download links using regex
        data_link_match = re.search(r"Data file available for download: (http[^\s]+)", response_text)
        metadata_link_match = re.search(r"Metadata file available for download: (http[^\s]+)", response_text)

        if data_link_match and metadata_link_match:
            return {
                "data_download_link": data_link_match.group(1),
                "metadata_download_link": metadata_link_match.group(1),
            }
        else:
            return {"error": "Download links not found in the response."}
    else:
        print(f"Request failed with status code {response.status_code}")
        print(response.text)
        return {"error": f"Request failed with status code {response.status_code}"}


In [ ]:
def download_file(download_link, token):
    # Define the headers with the JWT token
    headers = {
        "Authorization": f"Bearer {token}"
    }

    # Send a GET request to download the file
    response = requests.get(download_link, headers=headers)

    if response.status_code == 200:
        file_name = download_link.split("/")[-1]
        with open(file_name, "wb") as f:
            f.write(response.content)
        print(f"File downloaded successfully as {file_name}")
    else:
        print(f"Failed to download file: {response.status_code}")
        print(response.json())


## Execution

In [ ]:
# Log in and get the access token
token = login(username, password)

In [ ]:
token

In [ ]:
download_link = fetch_data(token)

In [ ]:
file = download_file('http://a52cbcc62931.ngrok-free.app/download/tmplsmoefqe.csv', token)

In [ ]:
file

In [ ]:
meta = download_file('http://a52cbcc62931.ngrok-free.app/download/tmpgyybklt2.json', token)

In [ ]:
meta